## 📝 Exercises

### Task 1
Parse text data into structured format

---

### Task 2
Count occurrences of each name

---

### Task 3
Filter invalid records safely

---

### Task 4
Calculate the average salary per department

In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install --upgrade pyspark


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pyspark.sql import SparkSession

# Re-initialize Spark with Java 17/25 bypass configurations
spark = SparkSession.builder \
    .appName("RDD_Lab_Exercises") \
    .master("local[*]") \
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow --add-exports=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-util=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED") \
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow --add-exports=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-util=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED") \
    .getOrCreate()

sc = spark.sparkContext

# Test loading the file again—it will bypass the error completely!
raw_rdd = sc.textFile("employees.txt")
print("Successfully loaded file into RDD!")

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
# Cell 1: Task 1 Implementation
def initial_split(line):
    """
    Basic split by comma and stripping whitespaces.
    Returns a list of raw string components.
    """
    return [part.strip() for part in line.split(',')]

# Apply the basic splitting to your cleaned raw text RDD
# (Assumes clean_raw_rdd is already defined and headers/empty lines are removed)
parsed_text_rdd = clean_raw_rdd.map(initial_split).cache()

print("--- Task 1 Complete: Text data parsed into raw lists ---")
print("Sample parsed row:", parsed_text_rdd.first())

In [ ]:
name_counts_rdd = valid_records_rdd \
    .map(lambda emp: (emp['name'], 1)) \
    .reduceByKey(lambda a, b: a + b)

print("\n--- Task 2: Name Counts (Sample) ---")
for name, count in name_counts_rdd.take(5):
    print(f"{name}: {count}")

In [ ]:
# Cell 2: Task 3 Implementation
import re

def validate_and_map(parts):
    """
    Validates the structure, repairs known data anomalies, 
    and returns a tuple: (is_valid, data_or_error_message)
    """
    if len(parts) < 8:
        return (False, f"Malformed row (too few columns): {parts}")
        
    try:
        emp_id = int(parts[0])
        name = parts[1]
        dept = parts[2]
        title = parts[3]
        salary_str = parts[4]
        
        # Repair logic for row 7 (missing comma between title and salary)
        if not salary_str.isdigit():
            merged_text = title + salary_str
            match = re.search(r'(.*??)(\d+)$', merged_text)
            if match:
                title = match.group(1).strip()
                salary_str = match.group(2)
            else:
                return (False, f"Invalid salary format: {salary_str}")
                
        salary = float(salary_str)
        
        # Readjust indices for rows with unexpected trailing columns
        if "Legal Counsel" in parts[3]:
            location = parts[4]
            hire_date = parts[5]
            perf_rating = float(parts[6])
            years_exp = int(parts[7])
        else:
            location = parts[5]
            hire_date = parts[6]
            perf_rating = float(parts[7])
            years_exp = int(parts[8])
            
        return (True, {
            "emp_id": emp_id, "name": name, "department": dept, 
            "job_title": title, "salary": salary, "location": location,
            "hire_date": hire_date, "performance_rating": perf_rating, "years_exp": years_exp
        })
        
    except Exception as e:
        return (False, f"Validation failed: {str(e)} on data {parts}")

# Map data through our validation filter
validated_results = parsed_text_rdd.map(validate_and_map).cache()

# Separate the dataset into Valid and Invalid RDDs
valid_records_rdd = validated_results.filter(lambda x: x[0] == True).map(lambda x: x[1])
invalid_records_rdd = validated_results.filter(lambda x: x[0] == False).map(lambda x: x[1])

print("--- Task 3 Complete: Safe Filtering ---")
print(f"Successfully validated records: {valid_records_rdd.count()}")
print(f"Safely filtered out invalid/malformed records: {invalid_records_rdd.count()}")

In [ ]:
# Step 1: Map to (Department, (Salary, 1))
dept_salary_tuples = valid_records_rdd.map(lambda emp: (emp['department'], (emp['salary'], 1)))

# Step 2: Reduce by key to get (Department, (Total_Salary, Total_Count))
dept_salary_totals = dept_salary_tuples.reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))

# Step 3: Map to calculate average (Department, Average_Salary)
dept_averages_rdd = dept_salary_totals.mapValues(lambda total_tuple: total_tuple[0] / total_tuple[1])

print("\n--- Task 4: Average Salary Per Department ---")
for dept, avg_sal in dept_averages_rdd.collect():
    print(f"Department: {dept:<12} | Average Salary: ${avg_sal:,.2f}")